# linearized model

$$ I_\textrm{b} \ddot{\theta} = m g l \theta - I_\textrm{w} \ddot{\varphi} $$

In [1]:
import numpy as np

m = 1
g = 9.8
l = 1
Iw = 1
Ib = 1

def eom_nonlinear(
    t: float,
    state: tuple[float, float, float],
) -> tuple[float, float, float]:
    theta, theta_dot, phi_dot = state
    phi_ddot = 0 # no motor controller input
    theta_ddot = (m * g * l * np.sin(theta) - Iw * phi_ddot) / Ib

    return theta_dot, theta_ddot, phi_ddot

In [2]:
from scipy.integrate import solve_ivp
import plotly.express as px
from IPython.display import display, HTML

# see https://stackoverflow.com/a/76599370
display(HTML(r'<script type="text/javascript" async src="https://cdnjs.cloudflare.com/ajax/libs/mathjax/2.7.1/MathJax.js?config=TeX-MML-AM_SVG"></script>'))


In [3]:
# nearly maximal pendulum swinging
theta_0 = 0.1
solution = solve_ivp(eom_nonlinear, t_span=[0, 10], y0=[theta_0, 0, 0], max_step=0.01)
assert solution.success
fig = px.scatter(x=solution.t, y=solution.y[0])
fig.update_layout(
    title=rf"$\text{{Nonlinear, }} \theta_0 = {theta_0}$",
    xaxis_title=r"$t$",
    yaxis_title=r"$\theta$",
)


In [4]:
# half pendulum swing
theta_0 = np.pi / 2
solution = solve_ivp(eom_nonlinear, t_span=[0, 10], y0=[theta_0, 0, 0], max_step=0.01)
assert solution.success
fig = px.scatter(x=solution.t, y=solution.y[0])
fig.update_layout(
    title=rf"$\text{{Nonlinear, }} \theta_0 = {theta_0}$",
    xaxis_title=r"$t$",
    yaxis_title=r"$\theta$",
)


In [5]:
# unstable equilibrium
theta_0 = 0
solution = solve_ivp(eom_nonlinear, t_span=[0, 5], y0=[theta_0, 0, 0], max_step=0.01)
assert solution.success

fig = px.scatter(x=solution.t, y=solution.y[0])
fig.update_layout(
    title=rf"$\text{{Nonlinear, }} \theta_0 = {theta_0}$",
    xaxis_title=r"$t$",
    yaxis_title=r"$\theta$",
)


In [6]:
# stable equilibrium
theta_0 = np.pi
solution = solve_ivp(eom_nonlinear, t_span=[0, 5], y0=[theta_0, 0, 0], max_step=0.01)
assert solution.success

fig = px.scatter(x=solution.t, y=solution.y[0])
fig.update_layout(
    title=rf"$\text{{Nonlinear, }} \theta_0 = {theta_0}$",
    xaxis_title=r"$t$",
    yaxis_title=r"$\theta$",
)


In [7]:
P = 20
D = 3

def eom_linear_pd(
    t: float,
    state: tuple[float, float, float],
) -> tuple[float, float, float]:
    theta, theta_dot, phi_dot = state
    phi_ddot = P * theta + D * theta_dot
    theta_ddot = (m * g * l * np.sin(theta) - Iw * phi_ddot) / Ib

    return theta_dot, theta_ddot, phi_ddot

theta_0 = 0.2
solution = solve_ivp(eom_linear_pd, t_span=[0, 10], y0=[theta_0, 0, 0], max_step=0.01)
assert solution.success

fig = px.scatter(x=solution.t, y=solution.y[0])
fig.update_layout(
    title=rf"$\text{{Linear, }} P={P}, D={D}, \theta_0={theta_0}$",
    xaxis_title=r"$t$",
    yaxis_title=r"$\theta$",
)

In [ ]:
import plotly.graph_objects as go

# plot speed as well

go.Figure(
    data=[
        go.Scatter(x=solution.t, y=solution.y[0], name=r"$\theta$"),
        go.Scatter(x=solution.t, y=solution.y[1], name=r"$\dot\theta$"),
    ],
    layout=go.Layout(
        title=rf"$\text{{Linear, }} P={P}, D={D}, \theta_0={theta_0}$",
        xaxis_title=r"$t$",
    )
)